In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import logging
from causalopt.thresh_tune import exc_optim_thresh
from causalopt.thresh_tradeoff import tradeoff_threshold
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO)

# Create Synthetic Data

For this example, let's create a simulated dataset. The idea is:
- We have a predicted probability from a hypothetical model (running variable)
- We set the threshold at 0.5. Because we draw these probabilities from an uniform distribution, this is also our median value.
- For customers with probability >= threshold, we recommend a certain intervention. Eg, a 10% discount in any product.
- This intervention makes users more likely to convert (increase A2C), but reduces the average price they pay (RPA).


In [ ]:
rng = np.random.default_rng(42)
n = 10000  # sample size
c = 0.5  # RD threshold
x = rng.uniform(0, 1, n)  # Probabilities
t = (x >= c).astype(int)  # Treatment

### First outcome - Completion
alpha_left = -1  # intercept when x < cutoff
alpha_right = 0.5  # intercept when x >= cutoff
beta_left = 0  # slope when x < cutoff
beta_right = 10  # slope when x >= cutoff
sigma_eps = 2.0  # std-dev of the Normal error term  ε ~ N(0, σ²)
epsilon = rng.normal(0, sigma_eps, n)
y = (
    (1 - t) * (alpha_left + beta_left * (x - c))
    + (t) * (alpha_right + beta_right * (x - c))
    + epsilon
)
y = (y >= 0).astype(int)

### Second outcome - Revenue
alpha_left = 100  # intercept when x < cutoff
alpha_right = 80  # intercept when x >= cutoff
beta_left = -150  # slope when x < cutoff
beta_right = -50  # slope when x >= cutoff
sigma_eps = 20.0  # std-dev of the Normal error term  ε ~ N(0, σ²)
epsilon = rng.normal(0, sigma_eps, n)
r = (
    (1 - t) * (alpha_left + beta_left * (x - c))
    + (t) * (alpha_right + beta_right * (x - c))
    + epsilon
)

df = pd.DataFrame(
    {
        "completion": y,  # vector for completion
        "revenue": r,  # vector for revenue
        "probability": x,  # vector of proabilities
        "threshold": c,  # center at c
    }
)

## Summary

In [ ]:
df.describe()

In [ ]:
df.loc[df["probability"] >= df["threshold"]].describe()

In [ ]:
df.loc[df["probability"] < df["threshold"]].describe()

## Density Plot

A common check for RDD is to look at how customers distributes along the running variable, particular close to the threshold. Note that there is no evidence of "manipulation", i.e., a high concentration of observations right above or right below the threshold. This is a good sign, indicating that users don't try to be "treated".

In [ ]:
prob = df["probability"]
ax = prob.plot.density(bw_method="scott", color="blue", linestyle="-", linewidth=2)
ax.axvline(x=0.5, color="r", linestyle="--", linewidth=1.5)
ax.set_xlim((0, 1))
ax.set_title("Probability Density and Threshold")

# Threshold Tuning

Now let's use rddthreshtuning to produce the results! The main function is get_optimum_threshold. For this function, we need to specify 4 arguments

- df = pandas dataframe we are using for the analysis.
- outcome: the name of the columns in df where our outcome is stored. In this example, we use completion and revenue.
- probability: the name of the column in df where our running variable is stored. Here it is the score probability from our ML model.
- thereshold: the name of the column in df where our threshold variable is stored. The threshold must be unique in each df. In this example, the variable threshold has a constant value of 0.5 for each customer.
- Optional: if verbose = True (default), the function will display a message indicating if we need to increase, decrease or keep the current threshold. Set verbose = False to omit the message.


In [ ]:
outcome_col = "completion"
probability_col = "probability"
threshold = 0.5

In [ ]:
results = exc_optim_thresh(df, outcome_col, probability_col, threshold)

# Analysis

Before looking at the outupt, here are guidelines to analyze the results.

1. Check for impact in the **main outcome** at threshold (eg, completed_flag).
  - If there is no impact, i.e., that estimate is not statistically significant, then **STOP** and **DO NOT CHANGE THE THRESHOLD**.
  - If there is impact, i.e., that estimate is statistically significant, then **GO TO STEP 2**.

2. Check if there's impact in the **alternative outcomes** at threshold (eg, total_revenue).
  - If there is no impact, i.e., that estimate is not statistically significant, then **GO TO STEP 3**.
  - If there is impact, i.e., that estimate is statistically significant, then **GO TO STEP 4**.

3. Use the threshold **recommendation provided in 1**.

4. Use the tradeoff function to evaluate the potential tradeoff between the **main** and the **alternative** outcomes.
  - If there is no tradeoff, i.e. the outcomes move in the same direction, use the threshold **recommendation in 1**.
  - If there is tradeoff, i.e. the outcomes move in the same direction, optimize the threshold by maximimzing the impact on the **main outcome** constrained by the maximum accepted loss in the **alternative outcomes**.



## Show RDD Results

In *rdrobust_results*, get_optimum_threshold stores the RDD results (estimated with rdrobust).

**Notes**
- As expected the policy increases the completion rate and decreases average revenue per user
- Just above the threshold, the completion rate *increases* by 28.7 p.p.
- Just above the threshold, the average revenue per use *decreases* by -$20.10.

In [ ]:
results["rd_results"]

## Plot RDD

In *graph_impact*, get_optimum_threshold stores a basic visualization for the RDD results

In [ ]:
from causalopt.plots import rdd_impact

rdd_impact(results, outcome_col)

## Threshold Recommendations

In *optimum_thresholds*, get_optimum_threshold stores two sets of arguments:
1. **Threshold**: optimum threshold values: optimum (median), a conservative, and an aggressive, respectively.
2. **Additional Gain**: the *additional expected grain* in the outcome variable if we set the optimum, the conservative, or the aggressive threshold, respectively.


In [ ]:
results["optimum_thresholds"]

## Threshold Plot

In [ ]:
from causalopt.plots import plot_thresh

plot_thresh(results, outcome_col)

# Tradeoff Functions

This is a classic example of a recommendation creating a tradeoff—where improving one metric (A2C) comes at the cost of another (RPA).
For instance, offering a discount to customers with a high likelihood of churning may increase A2C but decrease RPA. If we lower the threshold and extend the discount to more users, we drive more conversions but at the expense of revenue. Raising the threshold does the opposite—protecting revenue but reducing volume.
Our final decision will depend on how we choose to balance this tradeoff.

Let's use the function *threshold_tradeoff* to get estimates of gains and losses of each threshold. This function has 5 arguments.
1. **df** = pandas dataframe we are using for the analysis.
2. **outcomes**: the named of the columns in df where our outcome variables are stored. In this example, we use completion and revenue.
4. **probability**: the name of the column in df where our running variable is stored. Here it is the score probability from our ML model.
5. **thereshold**: the name of the column in df where our threshold variable is stored. The threshold must be unique in each df. In this example, the variable threshold has a constant value of 0.5 for each customer.


In [ ]:
results_tradeoff = tradeoff_threshold(df, ["completion", "revenue"], probability_col, threshold)

## Tradeoff Plot

In [ ]:
fig, ax1 = plt.subplots()

# Plot first variable
ax1.plot(
    results_tradeoff["x"],
    results_tradeoff["gain_completion"],
    color="blue",
    label="Completes Gain",
)
ax1.set_ylabel("Completes Gain", color="blue")
ax1.tick_params(axis="y", labelcolor="blue")

# Create a second y-axis sharing the same x-axis
ax2 = ax1.twinx()
ax2.plot(
    results_tradeoff["x"],
    results_tradeoff["gain_revenue"] / 1000,
    color="red",
    label="Revenue Gain",
)
ax2.set_ylabel("Revenue Gain (Thousands)", color="red")
ax2.tick_params(axis="y", labelcolor="red")

# Add x-axis label and title
_ = ax1.set_xlabel("Threshold Change")
plt.title("Completes and Revenue Gain vs. Threshold Change")